In [ ]:
## Import necessary modules
from sklearn import metrics
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve, matthews_corrcoef
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sn
import seaborn as sns
import pandas as pd
import csv
import math
from sklearn.feature_selection import SelectKBest, chi2
from sklearn import preprocessing
from sklearn.model_selection import cross_val_score,KFold,StratifiedKFold,cross_val_predict
from sklearn.metrics import classification_report
import sklearn
import tensorflow as tf
import keras
import numpy as np
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.models import Model
from keras.layers import LSTM, Dense, Bidirectional, LayerNormalization, BatchNormalization, Dropout
from keras.callbacks import ReduceLROnPlateau
from keras.regularizers import l2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pdv
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TP Prediction 2026/Features/AFP/AFPs_QLC_Training.csv')

In [ ]:
import numpy as np
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Replace inf values and handle missing values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(999, inplace=True)

# Split features and labels
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

print(f"Original data shape: {X.shape}")
print(f"Original class distribution: {Counter(y)}")

# =========================
# 1️⃣ Train-Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")
print(f"Training class distribution: {Counter(y_train)}")
print(f"Testing class distribution: {Counter(y_test)}")

# =========================
# 2️⃣ Standardization (Fit only on training data)
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from keras import layers, models
from keras.layers import LayerNormalization, Conv1D, Dropout, Flatten, Dense
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, roc_auc_score
import numpy as np
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
# -------------------
# MHA-Attention Block
# -------------------
def MultiHeadAttentionBlock(x, num_heads=4, key_dim=8, ff_dim=32, dropout_rate=0.2):
    # Multi-head self-attention (query=key=value=x)
    attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
    attn_output = Dropout(dropout_rate)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(x + attn_output)  # Residual

    # Feed-forward network
    ffn_output = Dense(ff_dim, activation='relu')(out1)
    ffn_output = Dense(out1.shape[-1])(ffn_output)  # project back to channels
    ffn_output = Dropout(dropout_rate)(ffn_output)
    out2 = LayerNormalization(epsilon=1e-6)(out1 + ffn_output)  # Residual
    return out2

# -------------------
# TCN Block
# -------------------
def TCNBlock(x, filters, kernel_size, dilation_rate):
    x = Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation_rate, activation='linear')(x)
    x = LayerNormalization()(x)
    x = layers.Activation('relu')(x)
    x = Dropout(0.2)(x)
    return x

# -------------------
# Hybrid TCN + Self-Attention Model
# -------------------
def create_bi_tcn_attention_model(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = TCNBlock(inputs, filters=64, kernel_size=1, dilation_rate=1)  # First block
    x = TCNBlock(x, filters=64, kernel_size=1, dilation_rate=2)  # Second block
    x = TCNBlock(x, filters=64, kernel_size=1, dilation_rate=4)  # 4th block
    #x = TCNBlock(x, filters=32, kernel_size=1, dilation_rate=8)  # 8th block

    # Residual connection
    x = layers.add([x, inputs])

    # Multi-Head Attention block (replaces self-attention name)
    x = MultiHeadAttentionBlock(x, num_heads=4, key_dim=8, ff_dim=32, dropout_rate=0.1)

    # Output
    x = Flatten()(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    return model

# -------------------
# Compile model
# -------------------
input_shape = (X_train.shape[1], 1)
model = create_bi_tcn_attention_model(input_shape)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# -------------------
# 10-Fold Cross-Validation
# -------------------
accuracy_list, f1_list, precision_list, recall_list = [], [], [], []
sensitivity_list, specificity_list, mcc_list, auc_list = [], [], [], []

# Perform K-fold cross-validation
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
from keras.callbacks import EarlyStopping, ReduceLROnPlateau


# 在K-fold循环中
for train_index, val_index in skf.split(X_train,y_train):
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    # 定义回调
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=5)

    callbacks = [early_stop, reduce_lr]

    # Train
    model.fit(X_train_fold, y_train_fold, epochs=40, batch_size=32, verbose=0, validation_data=(X_val_fold, y_val_fold), callbacks=callbacks)

    # Predict
    y_val_pred = model.predict(X_val_fold)
    y_val_pred_binary = (y_val_pred > 0.5).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y_val_fold, y_val_pred_binary)
    TN, FP, FN, TP = cm.ravel()

    # Metrics
    accuracy_list.append((TP + TN) / (TP + TN + FP + FN))
    f1_list.append(2 * TP / (2 * TP + FP + FN))
    precision_list.append(TP / (TP + FP))
    recall_list.append(TP / (TP + FN))
    sensitivity_list.append(TP / (TP + FN))
    specificity_list.append(TN / (TN + FP))
    mcc_list.append(((TP*TN) - (FP*FN)) / (np.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN)) or 1))
    auc_list.append(roc_auc_score(y_val_fold, y_val_pred))

# Average metrics
print("Accuracy =", np.mean(accuracy_list))
print("F1 Score =", np.mean(f1_list))
print("Precision =", np.mean(precision_list))
print("Recall =", np.mean(recall_list))
print("Sensitivity =", np.mean(sensitivity_list))
print("Specificity =", np.mean(specificity_list))
print("MCC =", np.mean(mcc_list))
print("AUC =", np.mean(auc_list))